# 오퍼 완료 예측 기반 추천 (ML)

## 목표
- **타깃**: 오퍼 수신 후 유효 기간(duration) 내 완료 여부
- **시퀀스/시간 반영**: 수신 시점(time), 고객별 오퍼 수신 순서
- **Cold start**: 신규 고객(프로필만), 신규 오퍼(오퍼 특성만) 대응
- **다양성·탐험**: 추천 리스트 내 오퍼 타입 다양성, ε-탐험
- **오프라인 메트릭**: AUC, Recall@k, NDCG, 다양성(엔트로피)

## 1. 환경 설정 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 (시각화 시)
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
if os.name == 'nt':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = ['AppleGothic', 'NanumGothic', 'sans-serif']

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CLEAN_DIR = "../데이터셋" if os.path.isdir("../데이터셋") else "../data/전처리_완료_데이터셋"
df = pd.read_csv(os.path.join(CLEAN_DIR, "starbucks_merged.csv"), encoding='utf-8-sig')
print(f"통합 데이터 shape: {df.shape}")
print(df['event'].value_counts())

통합 데이터 shape: (306066, 26)
event
transaction        138485
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64


## 2. 타깃 생성: 오퍼 수신 → 유효기간 내 완료 여부

In [2]:
# 오퍼 이벤트만 (transaction 제외)
offer_events = df[df['offer_id'].notna()].copy()
offer_events = offer_events.sort_values(['customer_id', 'time'])

# duration: 일 단위 → 시간 단위 (transcript time이 시간 단위라고 가정)
offer_events['duration_hours'] = offer_events['duration'].fillna(0) * 24

def build_target(events):
    """각 (customer_id, offer_id, 수신시점)에 대해 유효기간 내 완료 여부 계산"""
    received = events[events['event'] == 'offer received'].copy()
    completed = events[events['event'] == 'offer completed']
    rows = []
    for _, row in received.iterrows():
        cid, oid, t0 = row['customer_id'], row['offer_id'], row['time']
        dur = row.get('duration_hours', 7*24)
        if pd.isna(dur) or dur <= 0:
            dur = 7 * 24
        t1 = t0 + dur
        comp = completed[(completed['customer_id'] == cid) & (completed['offer_id'] == oid) & 
                          (completed['time'] >= t0) & (completed['time'] <= t1)]
        rows.append({
            'customer_id': cid, 'offer_id': oid, 'time_received': t0,
            'completed': 1 if len(comp) > 0 else 0,
            'duration_hours': dur,
            'gender': row.get('gender'), 'age': row.get('age'), 'income': row.get('income'),
            'became_member_on': row.get('became_member_on'),
            'reward': row.get('reward'), 'difficulty': row.get('difficulty'), 'duration': row.get('duration'),
            'offer_type': row.get('offer_type'),
            'channel_email': row.get('channel_email'), 'channel_mobile': row.get('channel_mobile'),
            'channel_social': row.get('channel_social'), 'channel_web': row.get('channel_web')
        })
    return pd.DataFrame(rows)

target_df = build_target(offer_events)
print(f"(수신 이벤트 기준) 행 수: {len(target_df):,}")
print(f"완료 비율: {target_df['completed'].mean():.2%}")
print(target_df.head())

(수신 이벤트 기준) 행 수: 76,277
완료 비율: 44.09%
                        customer_id                          offer_id  \
0  0009655768c64bdeb2e877511632db8f  5a8bc65990b245e5a138643cd4eb9837   
1  0009655768c64bdeb2e877511632db8f  3f207df678b143eea3cee63160fa8bed   
2  0009655768c64bdeb2e877511632db8f  f19421c1d4aa40978ebb69ca19b0e20d   
3  0009655768c64bdeb2e877511632db8f  fafdcd668e3743c1bb461111dcafc2a4   
4  0009655768c64bdeb2e877511632db8f  2906b810c7d4411798c6938adc9daaa5   

   time_received  completed  duration_hours gender  age   income  \
0            168          0            72.0      M   33  72000.0   
1            336          0            96.0      M   33  72000.0   
2            408          1           120.0      M   33  72000.0   
3            504          1           240.0      M   33  72000.0   
4            576          1           168.0      M   33  72000.0   

  became_member_on  reward  difficulty  duration     offer_type  \
0       2017-04-21     0.0         0.0       3.

## 3. 시퀀스/시간 피처 추가

In [3]:
# 고객별 오퍼 수신 순서 (시퀀스)
target_df = target_df.sort_values(['customer_id', 'time_received'])
target_df['offer_order'] = target_df.groupby('customer_id').cumcount() + 1

# 시간 구간 (정규화): 전체 기간 대비 수신 시점
t_min, t_max = target_df['time_received'].min(), target_df['time_received'].max()
target_df['time_norm'] = (target_df['time_received'] - t_min) / (t_max - t_min + 1e-6)

# 회원 가입 경과일 (became_member_on → 숫자)
target_df['became_member_on'] = pd.to_numeric(target_df['became_member_on'], errors='coerce')
ref_date = target_df['became_member_on'].max()  # 최근일 기준
target_df['member_tenure_days'] = (ref_date - target_df['became_member_on']).clip(lower=0)
target_df['member_tenure_days'] = target_df['member_tenure_days'].fillna(0)

print(target_df[['customer_id', 'offer_id', 'time_received', 'offer_order', 'time_norm', 'member_tenure_days', 'completed']].head(10))

                        customer_id                          offer_id  \
0  0009655768c64bdeb2e877511632db8f  5a8bc65990b245e5a138643cd4eb9837   
1  0009655768c64bdeb2e877511632db8f  3f207df678b143eea3cee63160fa8bed   
2  0009655768c64bdeb2e877511632db8f  f19421c1d4aa40978ebb69ca19b0e20d   
3  0009655768c64bdeb2e877511632db8f  fafdcd668e3743c1bb461111dcafc2a4   
4  0009655768c64bdeb2e877511632db8f  2906b810c7d4411798c6938adc9daaa5   
5  00116118485d4dfda04fdbaba9a87b5c  f19421c1d4aa40978ebb69ca19b0e20d   
6  00116118485d4dfda04fdbaba9a87b5c  f19421c1d4aa40978ebb69ca19b0e20d   
7  0011e0d4e6b944f998e987f904e8c1e5  3f207df678b143eea3cee63160fa8bed   
8  0011e0d4e6b944f998e987f904e8c1e5  2298d6c36e964ae4a3e7e9706d1fb8c2   
9  0011e0d4e6b944f998e987f904e8c1e5  5a8bc65990b245e5a138643cd4eb9837   

   time_received  offer_order  time_norm  member_tenure_days  completed  
0            168            1   0.291667                 0.0          0  
1            336            2   0.583333        

## 4. 학습용 피처 행렬 구성

In [4]:
# 결측 제거 (모델 입력용)
work = target_df.dropna(subset=['age', 'income', 'gender']).copy()
work = work[work['age'] < 100]  # 118 등 이상치 제외

# 범주형 인코딩
for col in ['gender', 'offer_type']:
    le = LabelEncoder()
    work[col + '_enc'] = le.fit_transform(work[col].astype(str))

feature_cols = ['age', 'income', 'member_tenure_days', 'offer_order', 'time_norm',
               'reward', 'difficulty', 'duration', 'gender_enc', 'offer_type_enc',
               'channel_email', 'channel_mobile', 'channel_social', 'channel_web']
feature_cols = [c for c in feature_cols if c in work.columns]
work[feature_cols] = work[feature_cols].fillna(0)

X = work[feature_cols]
y = work['completed']
print(f"학습 샘플 수: {len(X):,}, 완료율: {y.mean():.2%}")
print("피처:", feature_cols)

학습 샘플 수: 66,427, 완료율: 48.83%
피처: ['age', 'income', 'member_tenure_days', 'offer_order', 'time_norm', 'reward', 'difficulty', 'duration', 'gender_enc', 'offer_type_enc', 'channel_email', 'channel_mobile', 'channel_social', 'channel_web']


## 5. 시간 기반 Train/Test 분할

In [5]:
# 시간 기준 80% train, 20% test (데이터 누수 방지)
time_cut = work['time_received'].quantile(0.8)
train_idx = work['time_received'] < time_cut
test_idx = work['time_received'] >= time_cut

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {len(X_train):,}, Test: {len(X_test):,}")

Train: 44,239, Test: 22,188


## 6. 모델 학습 및 오프라인 메트릭

In [6]:
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
model.fit(X_train_s, y_train)

y_pred_proba = model.predict_proba(X_test_s)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
rec = recall_score(y_test, y_pred, zero_division=0)
prec = precision_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("=== 오프라인 메트릭 (Test) ===")
print(f"  AUC:       {auc:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  F1:        {f1:.4f}")

=== 오프라인 메트릭 (Test) ===
  AUC:       0.8147
  Recall:    0.8712
  Precision: 0.6830
  F1:        0.7657


## 7. Recall@k, NDCG, 다양성(엔트로피)

In [7]:
def recall_at_k(y_true, y_score, k=5):
    """고객·오퍼 단위가 아닌, 샘플 단위로 상위 k% 기준 recall"""
    n = len(y_true)
    n_pos = max(1, int(n * k / 100))
    order = np.argsort(-y_score)
    top = order[:n_pos]
    return y_true.iloc[top].sum() / max(1, y_true.sum())

def ndcg_at_k(y_true, y_score, k=5):
    from math import log2
    n = min(k, len(y_true))
    order = np.argsort(-y_score)[:n]
    dcg = sum((y_true.iloc[order].values / np.log2(np.arange(2, n+2))))
    ideal = np.sort(y_true.values)[::-1][:n]
    idcg = sum(ideal / np.log2(np.arange(2, n+2)))
    return dcg / idcg if idcg > 0 else 0

def diversity_entropy(offer_types_in_list):
    """추천 리스트 내 offer_type 분포 엔트로피"""
    from collections import Counter
    c = Counter(offer_types_in_list)
    n = sum(c.values())
    if n == 0: return 0
    return -sum((v/n)*np.log2(v/n) for v in c.values())

r5 = recall_at_k(y_test, y_pred_proba, k=5)
r10 = recall_at_k(y_test, y_pred_proba, k=10)
ndcg5 = ndcg_at_k(y_test, y_pred_proba, k=5)
print(f"Recall@5%:  {r5:.4f}")
print(f"Recall@10%: {r10:.4f}")
print(f"NDCG@5:     {ndcg5:.4f}")

Recall@5%:  0.0855
Recall@10%: 0.1642
NDCG@5:     1.0000


## 8. Cold start: 신규 고객 / 신규 오퍼

In [8]:
# 신규 고객: 과거 이벤트 없음 → 프로필 + 오퍼 특성만으로 예측 (동일 모델 사용)
# 신규 오퍼: 과거 완료 이력 없음 → 오퍼 특성 + 전역 평균 완료율로 스코어 부여

global_completion_rate = y_train.mean()
offer_avg = work.groupby('offer_id')['completed'].mean().to_dict()

def predict_cold_customer(profile_row, offer_features_df, model, scaler, feature_cols):
    """프로필 1건 + 모든 오퍼에 대해 완료 확률 예측 (신규 고객)"""
    # profile_row: age, income, gender_enc, member_tenure_days 등
    # offer_features_df: offer_id별 reward, difficulty, duration, offer_type_enc, channel_*
    rows = []
    for _, of in offer_features_df.iterrows():
        row = {**profile_row, **of}
        row = {k: row.get(k, 0) for k in feature_cols}
        rows.append([row.get(c, 0) for c in feature_cols])
    X = np.array(rows)
    X_s = scaler.transform(X)
    return model.predict_proba(X_s)[:, 1]

def score_cold_offer(offer_id, offer_avg, global_rate=0.2):
    """신규 오퍼: 학습 데이터에 없으면 global_rate 반환"""
    return offer_avg.get(offer_id, global_rate)

# 신규 고객 예시: 프로필만으로 모든 오퍼에 대해 예측 (동일 모델 사용)
customer_feature_cols = ['age', 'income', 'member_tenure_days', 'gender_enc', 'offer_order', 'time_norm']
offer_only_cols = ['reward', 'difficulty', 'duration', 'offer_type_enc', 'channel_email', 'channel_mobile', 'channel_social', 'channel_web']
offer_only_cols = [c for c in offer_only_cols if c in work.columns]
customer_feature_cols = [c for c in customer_feature_cols if c in work.columns]

offer_feature_df = work.drop_duplicates('offer_id')[['offer_id'] + offer_only_cols]
new_customer_example = work[customer_feature_cols].iloc[0].to_dict()
new_customer_example['offer_order'], new_customer_example['time_norm'] = 1, 0.5
X_new = pd.DataFrame([{**new_customer_example, **row} for _, row in offer_feature_df[offer_only_cols].iterrows()]).reindex(columns=feature_cols).fillna(0)
X_new_s = scaler.transform(X_new)
new_scores = model.predict_proba(X_new_s)[:, 1]
print("Cold start: 신규 고객 예시 - 모든 오퍼에 대한 완료 확률", new_scores.round(3))
print(f"전역 완료율(학습): {global_completion_rate:.2%}")

Cold start: 신규 고객 예시 - 모든 오퍼에 대한 완료 확률 [0.    0.    0.575 0.716 0.45  0.708 0.412 0.506 0.328 0.384]
전역 완료율(학습): 48.75%


## 9. 다양성·탐험 적용 추천 리스트 생성

In [9]:
# 고객별로 전체 오퍼에 대해 스코어링 후 상위 K개 추천 + ε 탐험
K = 3
EPSILON = 0.1  # 10% 확률로 k번째 슬롯을 무작위 오퍼로 대체 (탐험)

offer_ids = offer_feature_df['offer_id'].tolist()
offer_to_type = work.set_index('offer_id')['offer_type'].drop_duplicates().to_dict()

def recommend_topk(customer_id, pred_proba_by_offer, offer_ids, offer_to_type, k=K, epsilon=EPSILON):
    """상위 k개 선택, epsilon 확률로 마지막 슬롯을 무작위 오퍼로 대체 (탐험)"""
    order = np.argsort(-np.asarray(pred_proba_by_offer))
    chosen = []
    for i in order:
        if len(chosen) >= k:
            break
        oid = offer_ids[i] if isinstance(offer_ids[i], str) else str(offer_ids[i])
        if oid not in chosen:
            chosen.append(oid)
    if len(chosen) < k:
        for oid in offer_ids:
            if len(chosen) >= k: break
            oid = oid if isinstance(oid, str) else str(oid)
            if oid not in chosen:
                chosen.append(oid)
    if np.random.random() < epsilon and len(offer_ids) > k:
        others = [o for o in offer_ids if (o if isinstance(o, str) else str(o)) not in chosen]
        if others:
            chosen[-1] = others[np.random.randint(len(others))]
    return chosen[:k]

# 테스트 고객별로 모든 오퍼에 대해 예측 후 상위 K개 추천
test_customers = work[test_idx]['customer_id'].unique()
recs = []
for cid in test_customers[:500]:
    c_row = work[work['customer_id'] == cid][customer_feature_cols].drop_duplicates()
    if c_row.empty: continue
    c_row = c_row.iloc[0].to_dict()
    c_row['offer_order'], c_row['time_norm'] = 1, 0.5
    X_c = pd.DataFrame([{**c_row, **row} for _, row in offer_feature_df[offer_only_cols].iterrows()]).reindex(columns=feature_cols).fillna(0)
    X_c = scaler.transform(X_c)
    probs = model.predict_proba(X_c)[:, 1]
    top = recommend_topk(cid, probs, offer_ids, offer_to_type, k=K, epsilon=EPSILON)
    types_in = [offer_to_type.get(o, '') for o in top]
    recs.append({'customer_id': cid, 'recommended_offer_ids': top, 'diversity_entropy': diversity_entropy(types_in)})

rec_df = pd.DataFrame(recs)
print(f"추천 리스트 샘플 수: {len(rec_df)}")
print(f"평균 다양성(엔트로피): {rec_df['diversity_entropy'].mean():.4f}")
print(rec_df.head())

추천 리스트 샘플 수: 500
평균 다양성(엔트로피): 1.0530
                        customer_id  \
0  0009655768c64bdeb2e877511632db8f   
1  0011e0d4e6b944f998e987f904e8c1e5   
2  0020c2b971eb4e9188eac86d93036a77   
3  0020ccbbb6d84e358d3414a3ff76cffd   
4  003d66b6608740288d6cc97a6903f4f0   

                               recommended_offer_ids  diversity_entropy  
0  [fafdcd668e3743c1bb461111dcafc2a4, 2298d6c36e9...           1.584963  
1  [fafdcd668e3743c1bb461111dcafc2a4, 2298d6c36e9...           1.584963  
2  [fafdcd668e3743c1bb461111dcafc2a4, 2298d6c36e9...           0.918296  
3  [fafdcd668e3743c1bb461111dcafc2a4, 2298d6c36e9...           0.918296  
4  [fafdcd668e3743c1bb461111dcafc2a4, 2298d6c36e9...           0.918296  


## 10. 추천 결과 저장 및 요약

In [10]:
os.makedirs(CLEAN_DIR, exist_ok=True)

# 고객별 상위 오퍼 ID 리스트 (문자열로 저장)
rec_df['recommended_offer_ids_str'] = rec_df['recommended_offer_ids'].apply(lambda x: '|'.join(x))
out_cols = ['customer_id', 'recommended_offer_ids_str', 'diversity_entropy']
rec_df[out_cols].to_csv(os.path.join(CLEAN_DIR, 'offer_recommendations.csv'), index=False, encoding='utf-8-sig')

print("저장: offer_recommendations.csv")
print("\n=== 오프라인 메트릭 요약 ===")
print(f"  AUC:        {auc:.4f}")
print(f"  Recall@5%:  {r5:.4f}")
print(f"  NDCG@5:     {ndcg5:.4f}")
print(f"  다양성(평균 엔트로피): {rec_df['diversity_entropy'].mean():.4f}")

저장: offer_recommendations.csv

=== 오프라인 메트릭 요약 ===
  AUC:        0.8147
  Recall@5%:  0.0855
  NDCG@5:     1.0000
  다양성(평균 엔트로피): 1.0530
